# 06 — Ablation: mỗi lựa chọn đáng giá bao nhiêu?

**Chủ: Phong Nguyễn tổng hợp** · số liệu do Hoàng (components, scaling, label smoothing),
Phong Trần (resolution) và Huy (augmentation, preprocess) sinh ra

## ★ Nguyên tắc sắt: MỘT BIẾN MỘT LẦN

Đổi hai biến cùng lúc thì không quy được kết quả về nguyên nhân nào, và còn có thể bị
**triệt tiêu**: biến thứ nhất +0,3 điểm, biến thứ hai −0,3 điểm, tổng ra 0 và ta kết
luận **sai** rằng "cả hai đều vô dụng".

## 6 trục, 30 thực nghiệm

| Trục | Giá trị | Ai chạy |
|---|---|---|
| 1. resolution | 32/48/64 (M1,M2) · 64/112/224 (M3) | Phong Trần |
| 2. augmentation | none / geo / geo_photo / geo_photo_erase | Huy |
| 3. preprocess | none / he_gray / he_y / clahe | Huy |
| 4. scaling | width ×0,5/×1/×2 · depth 3/4/5 stage | Hoàng |
| 5. label smoothing | 0,0 / 0,1 / 0,2 | Hoàng |
| 6. components | bỏ BN / bỏ residual / bỏ spatial dropout | Hoàng |

```bash
python scripts/run_ablation.py --axes all --dry-run   # xem sẽ chạy gì
python scripts/run_ablation.py --axes all             # chạy thật (lâu)
python scripts/evaluate.py --runs "artifacts/runs/*"  # đo trên test
python scripts/run_ablation.py --collect-only         # tổng hợp bảng
```

In [ ]:
# Cho notebook thấy src/.  Chạy ô này đầu tiên.
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import os; os.chdir(ROOT)          # mọi đường dẫn trong repo là tương đối từ gốc

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from gtsrb import CLASS_NAMES, NUM_CLASSES
print("torch", torch.__version__, "| MPS", torch.backends.mps.is_available(),
      "| CUDA", torch.cuda.is_available())

In [ ]:
path = pathlib.Path("reports/tables/ablation.csv")
if not path.exists():
    print("Chưa có reports/tables/ablation.csv")
    print("Chạy: python scripts/run_ablation.py --axes all")
    print("      python scripts/evaluate.py --runs \"artifacts/runs/*\"")
    print("      python scripts/run_ablation.py --collect-only")
    abl = pd.DataFrame()
else:
    abl = pd.read_csv(path)
    abl["axis"] = abl["tag"].astype(str).str.split("-").str[0]
    print(f"{len(abl)} run, các trục: {sorted(abl['axis'].unique())}")
    abl.head()

In [ ]:
def plot_axis(axis_name, x_from_tag, xlabel, metric="test_macro_f1"):
    """Vẽ một trục ablation. x_from_tag: hàm lấy giá trị biến từ cột tag."""
    if abl.empty:
        print("chưa có số liệu"); return
    part = abl[abl["axis"] == axis_name].copy()
    if part.empty:
        print(f"chưa có run nào thuộc trục {axis_name!r}"); return
    part["x"] = part["tag"].map(x_from_tag)
    part = part.dropna(subset=[metric]).sort_values("x")
    if part.empty:
        print(f"trục {axis_name!r} chưa có số TEST — chạy scripts/evaluate.py"); return

    fig, ax = plt.subplots(figsize=(7, 3.8))
    for model, grp in part.groupby("model"):
        ax.plot(grp["x"].astype(str), grp[metric], marker="o", label=model)
    ax.set_xlabel(xlabel); ax.set_ylabel(metric)
    ax.set_title(f"Ablation: {axis_name}"); ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()

    print(part[["model", "tag", "test_top1", metric, "test_ece"]].to_string(index=False))
    spread = part[metric].max() - part[metric].min()
    print(f"\n>>> Trục {axis_name!r} làm {metric} thay đổi {spread:.4f} "
          f"({spread*100:.2f} điểm). Tốt nhất: {part.loc[part[metric].idxmax(),'tag']}")

## Trục 1 — độ phân giải đầu vào

In [ ]:
plot_axis("resolution", lambda t: int(str(t).split("res")[-1]) if "res" in str(t) else None,
          "img_size")

**Kỳ vọng.** Tăng độ phân giải giúp nhất ở các lớp mà **chữ số/chi tiết nhỏ** là
đặc trưng phân biệt (biển giới hạn tốc độ — xem `notebooks/05`). Nếu trục này cho
cải thiện rõ thì đó là bằng chứng rằng lỗi của nhóm là **vấn đề độ phân giải**, không
phải vấn đề dung lượng model.

## Trục 2 — augmentation

In [ ]:
plot_axis("augmentation", lambda t: str(t).replace("aug_", ""), "chính sách augment")

**Nhắc lại hai phép BỊ CẤM** (`notebooks/01` mục 4):
`RandomHorizontalFlip` (4 cặp lớp là ảnh gương → sai nhãn) và hue jitter (màu mang nghĩa).

Nếu `geo_photo_erase` **không** tốt hơn `geo_photo` thì kết luận là: augmentation
đã tới điểm bão hoà, thêm nữa không giúp. Đó cũng là một kết quả.

## Trục 3 — tiền xử lý (cân bằng sáng)

In [ ]:
plot_axis("preprocess", lambda t: str(t).replace("prep_", ""), "chế độ tiền xử lý")

## Trục 4 — model scaling

In [ ]:
plot_axis("scaling", lambda t: str(t), "cấu hình")

**Câu hỏi thật của trục này:** làm model **to hơn** có giúp không?
Nếu width ×2 (4,92M tham số) **không** hơn width ×1 (1,24M) thì model đã đủ dung lượng,
và nút cổ chai nằm ở chỗ khác (độ phân giải, hoặc chính dữ liệu). Kết luận đó
quan trọng hơn việc tìm ra cấu hình thắng.

## Trục 5 — label smoothing (★ chú ý cột ECE)

In [ ]:
plot_axis("label_smoothing", lambda t: float(str(t).replace("ls", "")), "epsilon")
# Và xem riêng ECE — đây là chỗ thú vị nhất
if not abl.empty:
    ls = abl[abl["axis"] == "label_smoothing"].dropna(subset=["test_ece"])
    if len(ls):
        fig, ax = plt.subplots(figsize=(6.5, 3.6))
        ls = ls.assign(eps=ls["tag"].str.replace("ls", "").astype(float)).sort_values("eps")
        ax.plot(ls["eps"], ls["test_ece"], marker="o", color="crimson", label="ECE (càng nhỏ càng tốt)")
        ax2 = ax.twinx()
        ax2.plot(ls["eps"], ls["test_macro_f1"], marker="s", color="steelblue", label="macro-F1")
        ax.set_xlabel("label smoothing epsilon"); ax.set_ylabel("ECE", color="crimson")
        ax2.set_ylabel("macro-F1", color="steelblue")
        ax.set_title("Đánh đổi: hiệu chỉnh vs accuracy"); ax.grid(alpha=.3)
        plt.tight_layout(); plt.show()
        print(ls[["tag","test_top1","test_macro_f1","test_ece"]].to_string(index=False))

**★ Giả thuyết của nhóm — và nó NGƯỢC với lý thuyết thông thường.**

`notebooks/05` mục 4 đo được: M1 với `ε = 0,1` cho `conf = 0,854 < acc = 0,985`
→ model **THIẾU tự tin**, ECE = 0,131 (cao).

Lý thuyết (Guo 2017) nói mạng sâu tự tin thái quá và label smoothing sửa được — nhưng
kết luận đó đúng trên ImageNet (bài toán **khó**, acc ~76%). GTSRB **quá dễ** (acc 98,5%),
nên mức tự tin mà label smoothing kéo về (≈ 0,9023) **thấp hơn** accuracy thật
→ nó **sửa quá tay**.

**Giả thuyết: `ε = 0,0` sẽ cho ECE TỐT HƠN trên GTSRB.**
Trục này kiểm chứng. Nếu đúng, đó là kết quả đáng báo cáo nhất của phần ablation.

## Trục 6 — từng thành phần kiến trúc của M2

In [ ]:
plot_axis("components", lambda t: str(t), "biến thể")

Trục này trả lời trực tiếp: **BatchNorm / residual / spatial dropout đáng giá bao nhiêu?**
Đây là bằng chứng cho phần trình bày của Hoàng — không nói theo bài báo, mà đo trên
chính dữ liệu của nhóm.

## Tổng hợp: trục nào đáng giá nhất?

In [ ]:
if not abl.empty and abl["test_macro_f1"].notna().any():
    summary = (abl.dropna(subset=["test_macro_f1"])
                  .groupby("axis")["test_macro_f1"]
                  .agg(["min", "max", "count"]))
    summary["biên độ"] = summary["max"] - summary["min"]
    summary = summary.sort_values("biên độ", ascending=False)
    print(summary.round(4).to_string())
    print()
    print("Biên độ lớn  = biến này QUAN TRỌNG, đáng đầu tư thời gian.")
    print("Biên độ nhỏ  = biến này không quan trọng -> kết luận 'không cần tinh chỉnh'")
    print("               cũng là một kết quả có giá trị, đừng bỏ qua.")
else:
    print("Chưa có đủ số liệu test. Chạy: make eval")